In [1]:
# execute as if in root folder for utils and filepaths
%cd ..

c:\Users\sonja\OneDrive\Uni\Semester 6\BachelorArbeit\practical-work-ai-bsc


C:\Users\sonja\AppData\Roaming\Python\Python311\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
import numpy as np
import random
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import json
from torch.utils.tensorboard import SummaryWriter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, auc, precision_recall_curve , average_precision_score
from torch.optim.lr_scheduler import ReduceLROnPlateau


from utils import get_support_query_loaders_ep, log_results_nn_ft, NeuralNetwork, save_best_model, classification_loss, propagation_loss, getPrAucIndividualClassFinetuning

In [3]:
data = np.load('./data/data_scaled.npz')
X_finetuning = data['X_finetuning']
y_ft1 = data['y_finetuning1']
y_ft2 = data['y_finetuning2']

In [4]:
seeds = [2608, 2831, 3525, 3127, 9549, 9299, 896, 8017, 106, 7039]

In [5]:
def label_propagation(X, Y, alpha):
    """
    X: input features
    Y: one hot vector of n x c dimensions; c is num of different labels
    alpha: hyperparameter that controls how much influence neighbors get, [0,1]: 0: predict only 0s, 1: predict only 1s
    """
    device = X.device
    # affinity matrix W
    sq = torch.sum(X ** 2, dim=1, keepdim=True)  # shape: (num_samples, 1)
    distances_squared = sq + sq.t() - 2 * (X @ X.t())
    sigma2 = torch.var(distances_squared)
    W = torch.exp(-distances_squared / (2*sigma2) )
    W.fill_diagonal_(0)

    # diagonal matrix
    row_sums = torch.sum(W, dim=1)
    D12 = torch.diag(1 / torch.sqrt(row_sums))
    
    S = D12 @ W @ D12

    # F*
    I = torch.eye(X.shape[0], device=device, dtype=X.dtype)
    temp = (I - alpha * S).float()
    F_star = torch.linalg.solve(temp, Y) # solve MF = Y for F, avoid direct inverse

    # label each point x_i as y_i = argamx F*_ij
    L = torch.argmax(F_star, dim=1)

    return L


In [6]:
def prepare_labels_label_propagation(len_X_support, len_X_query, y_support, device, num_classes=2):
    y_support = y_support.clone().detach().long()
    zeros = torch.zeros((len_X_query, num_classes), dtype=torch.float, device=device)
    one_hot_y= nn.functional.one_hot(y_support, num_classes,).float().to(device)
    Y = torch.concatenate((one_hot_y, zeros))
    return Y

In [7]:
def scatterplot_propagated_labels(X, y1, y2):
    pca = PCA(n_components=2, random_state=0)
    X2 = pca.fit_transform(X)

    # Prepare subplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

    for ax, (y, title) in zip(axes, [(y1, 'Labels → y1'), (y2, 'Labels → y2 (None→grey)')]):
        # Find unique real classes (ignore None/nan)
        classes = [c for c in np.unique(y) if c is not None and not (isinstance(c, float) and np.isnan(c))]
        # Plot each real class with default cycle
        for cls in classes:
            mask = (y == cls)
            ax.scatter(
                X2[mask, 0], X2[mask, 1],
                label=f'Class {cls}',
                s=20, alpha=0.5, edgecolors='face'
            )
        # Plot missing (None/nan) as grey
        missing_mask = [(v is None) or (isinstance(v, float) and np.isnan(v)) for v in y]
        if any(missing_mask):
            ax.scatter(
                X2[missing_mask, 0], X2[missing_mask, 1],
                label='Missing',
                color='lightgrey',
                s=20, alpha=0.1, edgecolors='face'
            )
        ax.set_title(title, fontsize=16)
        ax.set_xlabel('PC1', fontsize=14)
        ax.set_ylabel('PC2', fontsize=14)
        ax.legend(title='Label', fontsize=12, title_fontsize=12, loc='best')
        ax.grid(True)

    plt.tight_layout()
    plt.show()

In [8]:
# task_idx = 2
# support_loader, _, support_idx, query_idx = get_support_query_loaders(X_finetuning, y_ft1, task_idx, k=1, l=30, max_pos_query=10, seed=12576, batch_size=120, num_workers=0)
# print(y_ft1[support_idx, task_idx])
# Y = prepare_labels_label_propagation(len(support_idx), len(query_idx), y_ft1[support_idx, 2])
# episode_idx = np.concatenate((support_idx, query_idx))
# propagated_labels = label_propagation(torch.tensor(X_finetuning[episode_idx], device="cuda"), torch.tensor(Y, device="cuda"), alpha=0.0000000000000001)

In [9]:
# scatterplot_propagated_labels(X_finetuning[episode_idx], propagated_labels.cpu(), y_ft1[episode_idx, task_idx])

In [10]:
def train_few_shot_model(model, X, y, task_idx, optimizer, scheduler, device, writer, run, k, l, max_pos_query, seed, alpha, alpha_ep, max_episodes=10, patience=100, batch_size=120, num_workers=0):
    """
    model          : your NeuralNetwork instance
    X, y           : numpy arrays (N, D) and (N, T)
    criterion      : loss fn, e.g. BCEWithLogitsLoss()
    optimizer      : your torch optimizer
    device         : torch.device
    writer         : tensorboard writer
    run            : an identifier for the saved filename
    k, seed        : few-shot support size and RNG seed
    """
    model.to(device)
    best_score = 0.0
    file_path = f"./best_models/ft1_ep_{run}.pth"

    X = torch.from_numpy(X).float().to(device)
    y = torch.from_numpy(y).float().to(device)

    epochs_no_improve = 0
    best_pr_auc = None
    best_roc_auc = None
    pr_auc_per_task = None
    roc_auc_per_task = None
    last_improvement_episode = 0

    # generate support and query sets
    support_loader, query_loader, support_idx, query_idx = get_support_query_loaders_ep(X, y, alpha_ep, task_idx, k, l, max_pos_query, seed, batch_size=batch_size, num_workers=num_workers)


    for episode in range(1, max_episodes + 1):
        model.train()
        ################# TRAIN #################
        # ——— SUPPORT SET ———
        for inputs, labels in support_loader:
            # flatten if needed
            support_inputs = inputs.view(inputs.size(0), -1).to(device)
            support_labels = labels.to(device)

            optimizer.zero_grad()
            outputs_support = model(support_inputs)
            outputs_support = outputs_support.squeeze(1)

            loss_support = classification_loss(outputs_support, support_labels)

        # ——— LOSS AND UPDATE STEP ——— # 
        optimizer.zero_grad()
        loss_support.backward()
        optimizer.step()
        scheduler.step(loss_support)

        ################# EVAL #################
        # ——— QUERY SET ——— #
        model.eval()
        all_labels_val = []
        all_preds_val = []

        for inputs, labels in query_loader:
            query_inputs = inputs.to(device)
            query_labels = labels.to(device)

            outputs_query = model(query_inputs)
            outputs_query = outputs_query.squeeze(1)

            loss_query = classification_loss(outputs_query, query_labels)

            all_labels_val.append(labels.cpu().numpy())
            all_preds_val.append(torch.sigmoid(outputs_query).detach().cpu().numpy())


        # ——— LABEL PROPAGATION ——— # 
        episode_idx = np.concatenate((support_idx, query_idx))
        Y = prepare_labels_label_propagation(len(support_idx), len(query_idx), y[support_idx, task_idx], device)
        propagated_labels = label_propagation(X[episode_idx], Y, alpha=alpha)
        query_logits = propagated_labels[len(support_idx):]
        loss_label_prop = propagation_loss(query_logits.float(), query_labels.float())

        # ——— EVALUATE MODEL ——— # 
        final_preds = (all_preds_val[0] + np.array(query_logits.cpu())) / 2 # final predictions are avg of model preds and label prop preds
        val_loss = classification_loss(torch.tensor(final_preds), torch.tensor(all_labels_val[0]))
        #test = [final_preds.reshape(-1)]
        val_pr_auc, val_roc_auc, val_pr_auc_per_task, val_roc_auc_per_task = getPrAucIndividualClassFinetuning(
            "Val", all_labels_val, [final_preds.reshape(-1)], writer, episode
        )

        # ——— SAVE BEST MODEL ——— # 
        saved = save_best_model(model, avg_pr_auc=val_pr_auc,best_score=best_score, file_path=file_path)
        if saved:
            best_score = val_pr_auc
            epochs_no_improve = 0
            last_improvement_episode = episode
            # save results
            best_pr_auc = np.mean(val_pr_auc)
            best_roc_auc = np.mean(val_roc_auc)
            pr_auc_per_task = np.mean(val_pr_auc_per_task, axis=0)
            roc_auc_per_task = np.mean(val_roc_auc_per_task, axis=0)
        else:
            epochs_no_improve += 1

        # early stopping
        if epochs_no_improve >= patience:
            print(f"Stopping early at episode {episode} (no improvement in {patience} episodes)")
            break

    print(
        f"=== Best Model Evaluation ===\n"
        f"Avg PR-AUC: {best_pr_auc:.4f}, Avg ROC-AUC: {best_roc_auc:.4f}"
    )

    return best_pr_auc, best_roc_auc, pr_auc_per_task, roc_auc_per_task, last_improvement_episode


In [11]:
# hyperparameters & global vars
input_size = 2248 
hidden_sizes = [120, 48, None]
output_size = 9
k = 1
batch_size = 500
dropout_rates = [0.3, 0.5] # input dropout, hidden dropout
lr = 0.2941271118953339
l = 10
max_pos_query= int(l/2)
max_episodes = 1205
patience = 282
alpha = 0.2538481528053722
alpha_ep = 0.8482506849337167
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# list to collect results avg over task
avg_results = {
        "Delta-AUC-PR": [],
        "ROC-AUC": []
    }

# list to collect results per task
all_results = {
        "Delta-AUC-PR": [],
        "ROC-AUC": []
    }  

num_tasks = 4
num_runs  = 5
last_improvement_episodes = np.zeros((num_tasks, num_runs))

# train and evaluate model 5 times
for task_idx in range(4):
    for run in range(5):
        # set seed
        seed_value = seeds[run]
        torch.manual_seed(seed_value)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed_value)
            torch.cuda.manual_seed_all(seed_value)
        np.random.seed(seed_value)
        random.seed(seed_value)

        # initialize
        # writer = SummaryWriter(f"manual_runs/run_36_upsampling") # tensorboard
        writer = None
        model = NeuralNetwork(input_size, hidden_sizes, output_size, dropout_rates)

        # load best model
        state_dict = torch.load("./best_models/overall_best_ep.pth", map_location=device)
        model.load_state_dict(state_dict)

        # freeze all parameters except the last layer (fc3)
        for name, param in model.named_parameters():
            if not name.startswith("fc3"):
                param.requires_grad = False

        # check if all frozen:
        for name, param in model.named_parameters():
            print(name, param.requires_grad)
            
        # replace fc3 with a new Linear that has a single output neuron
        in_features = model.fc3.in_features # keep input features as they are
        model.fc3 = nn.Linear(in_features, 1) # replace output features
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr) # filter out all parameters except last layer
        scheduler = ReduceLROnPlateau(optimizer,
                                mode='min',
                                factor=0.1,
                                patience=20,
                                min_lr=1e-6)

        # train model
        val_delta_auc_pr, val_roc_auc, val_pr_auc_per_task, val_roc_auc_per_task, last_improvement_episode = episode = train_few_shot_model(model, X_finetuning, y_ft1, task_idx, optimizer, scheduler, device, writer, run, k, l, max_pos_query, seed_value, alpha, alpha_ep, max_episodes, patience, batch_size=batch_size, num_workers=0)
        
        # collect results per run
        all_results["Delta-AUC-PR"].append(val_delta_auc_pr)
        all_results["ROC-AUC"].append(val_roc_auc) 

        last_improvement_episodes[task_idx, run] = last_improvement_episode

cuda
fc1.weight False
fc1.bias False
fc2.weight False
fc2.bias False
fc3.weight True
fc3.bias True
torch.Size([10]) torch.Size([10])
Saved new best model with avg PR AUC: 0.2500
torch.Size([10]) torch.Size([10])
Current avg PR AUC: -0.1353 did not improve over best score: 0.2500
torch.Size([10]) torch.Size([10])
Current avg PR AUC: -0.1956 did not improve over best score: 0.2500
torch.Size([10]) torch.Size([10])
Current avg PR AUC: 0.2500 did not improve over best score: 0.2500
torch.Size([10]) torch.Size([10])
Current avg PR AUC: 0.2500 did not improve over best score: 0.2500
torch.Size([10]) torch.Size([10])
Current avg PR AUC: 0.2500 did not improve over best score: 0.2500
torch.Size([10]) torch.Size([10])
Current avg PR AUC: 0.2500 did not improve over best score: 0.2500
torch.Size([10]) torch.Size([10])
Current avg PR AUC: 0.2500 did not improve over best score: 0.2500
torch.Size([10]) torch.Size([10])
Current avg PR AUC: 0.1722 did not improve over best score: 0.2500
torch.Size([

In [12]:
filepath_normal = "./metrics/ep_ft1.json"
filepath_per_task = "./metrics/ep_ft1_per_task.json"
filepath_per_run = "./metrics/ep_ft1_per_run.json"
log_results_nn_ft(all_results, filepath_normal, filepath_per_run, filepath_per_task)

In [13]:
last_improvement_episodes

array([[ 55., 181.,   1.,   1.,   9.],
       [ 43.,   9.,   7.,   5.,  53.],
       [ 10.,  84.,  10.,  12.,  27.],
       [ 46.,  48.,   7.,  33.,   5.]])

In [14]:
np.median(last_improvement_episodes)

11.0